In [ ]:
import json,pickle,zipfile,re,math,os,time
import numpy as np
import unicodedata
import torch, torch.nn as nn, torch.nn.functional as F
from peft import LoraConfig,AdaLoraConfig, get_peft_model
from collections import Counter
from tqdm import tqdm
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from torch.utils.data import DataLoader
from torch.utils import checkpoint
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

In [ ]:
## tokenizer
 
WORD_END = 256
 
def pre_tokenize_preserve(text):
    return re.findall(r"\w+|[^\w\s]|\s+", text)
 
def word_to_symbols(word):
    byte_values = word.encode("utf-8")
    symbols = [(b,) for b in byte_values]
    symbols.append((WORD_END,))
    return tuple(symbols)
 
def get_adjacent_pairs(symbols):
    return set(zip(symbols, symbols[1:]))
 
def apply_bpe_merges(symbols, merge_rank):
    symbols = list(symbols)
    while len(symbols) > 1:
        pairs = get_adjacent_pairs(symbols)
        candidates = [p for p in pairs if p in merge_rank]
        if not candidates:
            break
        best_pair = min(candidates, key=lambda p: merge_rank[p])
        sa, sb = best_pair
        merged = sa + sb
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == sa and symbols[i + 1] == sb:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols
 
def load_tokenizer(pkl_path):
    bpe_data = pickle.load(open(pkl_path, "rb"))
    stoi, merge_rank = bpe_data["stoi"], bpe_data["merge_rank"]
    cache = {}
 
    def encode_unit(unit):
        if unit not in cache:
            cache[unit] = [stoi[s] for s in apply_bpe_merges(word_to_symbols(unit), merge_rank)]
        return cache[unit]
 
    def encode(text):
        ids = []
        for unit in pre_tokenize_preserve(text):
            ids += encode_unit(" " if unit.isspace() else unit)
        return ids
 
    return bpe_data, encode
 
 
def clean_text(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"(\w)-[ \t]*\n[ \t]*(\w)", r"\1\2", t)
    return re.sub(r"\s+", " ", t).strip()

# MODEL V2

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, context_length, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RoPE(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.zeros_(module.bias)
            nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V5

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present

class Expert(nn.Module):
    def __init__(self, n_embd, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        hidden = expert_hidden_mult * n_embd
        self.c_fc = nn.Linear(n_embd, hidden)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(hidden, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class MoEFeedForward(nn.Module):
    def __init__(self, n_embd, num_experts=4, expert_hidden_mult=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.router = nn.Linear(n_embd, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(n_embd, expert_hidden_mult, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)                                  

        router_logits = self.router(x_flat)                     
        router_probs = F.softmax(router_logits, dim=-1)
        top1_prob, top1_idx = router_probs.max(dim=-1)          
        one_hot = F.one_hot(top1_idx, num_classes=self.num_experts).to(x.dtype)   

        expert_outputs = torch.stack([expert(x_flat) for expert in self.experts], dim=1)  
        gate = (one_hot * top1_prob.unsqueeze(-1)).unsqueeze(-1)                          
        output = (expert_outputs * gate).sum(dim=1)                                        
        with torch.no_grad():
            tokens_per_expert = one_hot.float().mean(dim=0)      
        mean_router_prob = router_probs.mean(dim=0)              
        aux_loss = self.num_experts * (tokens_per_expert * mean_router_prob).sum()

        return output.view(B, T, C), aux_loss

class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, num_experts, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_1_post = RMSNorm(n_embd)

        self.ln_2_pre = RMSNorm(n_embd)
        self.moe = MoEFeedForward(n_embd, num_experts=num_experts, dropout=dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward_train(self, x, attn_mask):
        attn_out, _ = self.attn(self.ln_1_pre(x), attn_mask)
        x = x + self.ln_1_post(attn_out)

        moe_out, aux_loss = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, aux_loss

    def forward_infer(self, x, past_kv, use_cache, position_offset):
        attn_out, present = self.attn(self.ln_1_pre(x), None, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)

        moe_out, _ = self.moe(self.ln_2_pre(x))
        x = x + self.ln_2_post(moe_out)
        return x, present

class DecoderOnlyLM_MoE_Sandwich(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,num_experts=4, aux_loss_weight=0.01, dropout=0.1, pad_id=0,gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.aux_loss_weight = aux_loss_weight
        self.gradient_checkpointing = gradient_checkpointing

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, num_experts, dropout)
            for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        x = self.drop(self.wte(input_ids))

        if use_cache:
            presents = []
            for i, block in enumerate(self.blocks):
                past = past_key_values[i] if past_key_values is not None else None
                x, present = block.forward_infer(x, past, use_cache, past_len)
                presents.append(present)
        else:
            combined_mask = None
            if attention_mask is not None:
                key_mask = attention_mask.bool()[:, None, None, :]
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]

            aux_losses = []
            for block in self.blocks:
                if self.gradient_checkpointing and self.training:
                    x, aux = checkpoint.checkpoint(block.forward_train, x, combined_mask,
                                                   use_reentrant=False)
                else:
                    x, aux = block.forward_train(x, combined_mask)
                aux_losses.append(aux)
            presents = None

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            ce_loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
            aux_loss = torch.stack(aux_losses).mean() if not use_cache else torch.tensor(0.0, device=x.device)
            loss = ce_loss + self.aux_loss_weight * aux_loss

        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=None,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_active_parameters(model, num_experts):
    total = count_parameters(model)
    expert_params = sum(p.numel() for p in model.blocks[0].moe.experts[0].parameters())
    num_layers = len(model.blocks)
    unused_experts_params = expert_params * (num_experts - 1) * num_layers
    return total - unused_experts_params

# MODEL V6

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)


def is_global_layer(layer_idx, n_layer, global_every_n):
    return (n_layer - 1 - layer_idx) % global_every_n == 0


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size=None, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head
        self.window_size = window_size   

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def _build_mask(self, T, device, key_padding_mask=None):
        i = torch.arange(T, device=device)
        if self.window_size is None:
            causal = i[:, None] >= i[None, :]                              
        else:
            diff = i[:, None] - i[None, :]
            causal = (diff >= 0) & (diff < self.window_size)               
        mask = causal[None, None, :, :]
        if key_padding_mask is not None:
            key_mask = key_padding_mask.bool()[:, None, None, :]
            eye = torch.eye(T, dtype=torch.bool, device=device)
            mask = (mask & key_mask) | eye[None, None, :, :]
        return mask

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
            if self.window_size is not None and k.size(2) > self.window_size:
                k = k[:, :, -self.window_size:, :]
                v = v[:, :, -self.window_size:, :]
                if key_padding_mask is not None:
                    key_padding_mask = key_padding_mask[:, -self.window_size:]
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if past_kv is None and T > 1:
            attn_mask = self._build_mask(T, x.device, key_padding_mask)
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        else:
            attn_mask = None
            if key_padding_mask is not None:
                attn_mask = key_padding_mask.bool()[:, None, None, :]
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, window_size, dropout=0.1):
        super().__init__()
        self.ln_1_pre = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, window_size, dropout)
        self.ln_1_post = RMSNorm(n_embd)
        self.ln_2_pre = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
        self.ln_2_post = RMSNorm(n_embd)

    def forward(self, x, key_padding_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1_pre(x), key_padding_mask, past_kv, use_cache, position_offset)
        x = x + self.ln_1_post(attn_out)
        x = x + self.ln_2_post(self.mlp(self.ln_2_pre(x)))
        return (x, present) if use_cache else x

class DecoderOnlyLM_HybridMask(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                window_size=192, global_every_n=4, dropout=0.1, pad_id=0,
                gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.window_size = window_size
        self.global_every_n = global_every_n

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)

        layer_windows = [
            None if is_global_layer(i, n_layer, global_every_n) else window_size
            for i in range(n_layer)
        ]
        print(f"Layer attention pattern (None = global/full-causal): {layer_windows}")

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, layer_windows[i], dropout)
            for i in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        if attention_mask is None and past_key_values is None:
            attention_mask = input_ids.ne(self.pad_id)

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, attention_mask, None, False, past_len,
                                          use_reentrant=False)
            else:
                out = block(x, attention_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]
        running_mask = attention_mask

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            running_mask = torch.cat([running_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=running_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

# MODEL V7

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm_x * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE requires an even head_dim"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len, offset, device, dtype):
        needed = offset + seq_len
        if needed > self.cos_cached.shape[0]:
            self._build_cache(needed)
        cos = self.cos_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[offset:offset + seq_len].to(device=device, dtype=dtype)
        return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return (x * cos) + (rotate_half(x) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)

        self.attn_dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.rope = RotaryEmbedding(self.head_dim, context_length)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(T, position_offset, x.device, x.dtype)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None

        k_exp = k.repeat_interleave(self.n_rep, dim=1)
        v_exp = v.repeat_interleave(self.n_rep, dim=1)

        dropout_p = self.attn_dropout_p if self.training else 0.0
        if attn_mask is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, attn_mask=attn_mask, dropout_p=dropout_p)
        elif past_kv is not None:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p)
        else:
            y = F.scaled_dot_product_attention(q, k_exp, v_exp, dropout_p=dropout_p, is_causal=True)

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, context_length, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, context_length, dropout)
        self.ln_2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False, position_offset=0):
        attn_out, present = self.attn(self.ln_1(x), attn_mask, past_kv, use_cache, position_offset)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return (x, present) if use_cache else x

class DecoderOnlyLM_RMSNorm_GQA(nn.Module):
    def __init__(self, vocab_size, context_length, n_embd, n_layer, n_head, n_kv_head,
                dropout=0.1, pad_id=0, gradient_checkpointing=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.context_length = context_length
        self.pad_id = pad_id
        self.gradient_checkpointing = gradient_checkpointing
        self.wte = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, n_kv_head, context_length, dropout) for _ in range(n_layer)
        ])
        self.ln_f = RMSNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None,
                past_key_values=None, use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + T <= self.context_length, \
            f"seq len {past_len + T} > context_length {self.context_length}"

        combined_mask = None
        if attention_mask is not None:
            key_mask = attention_mask.bool()[:, None, None, :]
            if past_len == 0:
                causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device))
                eye = torch.eye(T, dtype=torch.bool, device=input_ids.device)
                combined_mask = (causal[None, None] & key_mask) | eye[None, None]
            else:
                combined_mask = key_mask

        x = self.drop(self.wte(input_ids))

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = past_key_values[i] if past_key_values is not None else None
            if self.gradient_checkpointing and self.training and not use_cache:
                x = checkpoint.checkpoint(block, x, combined_mask, None, False, past_len,use_reentrant=False)
            else:
                out = block(x, combined_mask, past, use_cache, past_len)
                if use_cache:
                    x, present = out
                    presents.append(present)
                else:
                    x = out

        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=self.pad_id,
            )
        return {"loss": loss, "logits": logits, "past_key_values": presents}

    def _sample_next_token(self, logits, temperature, top_p, top_k):
        logits = logits / max(temperature, 1e-5)
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_mask = cumulative - sorted_probs > top_p
            sorted_probs[sorted_mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(sorted_probs, num_samples=1)
            return torch.gather(sorted_idx, -1, next_token)
        return torch.multinomial(probs, num_samples=1)

    @torch.no_grad()
    def generate_batch(self, input_ids, attention_mask, max_new_tokens,
                        temperature=0.7, top_p=0.95, top_k=50, eos_id=None):
        self.eval()
        B, T0 = input_ids.shape
        assert T0 + max_new_tokens <= self.context_length, "prompt + new tokens exceed context_length"
        finished = torch.zeros(B, dtype=torch.bool, device=input_ids.device)

        out = self.forward(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out["past_key_values"], out["logits"][:, -1, :]

        for step in range(max_new_tokens):
            next_token = self._sample_next_token(logits, temperature, top_p, top_k)

            if eos_id is not None:
                next_token = torch.where(
                    finished.unsqueeze(1), torch.full_like(next_token, eos_id), next_token
                )

            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

            if eos_id is not None:
                finished = finished | (next_token.squeeze(1) == eos_id)
                if finished.all():
                    break
            if step == max_new_tokens - 1:
                break

            out = self.forward(next_token, attention_mask=attention_mask,
                               past_key_values=past, use_cache=True, last_only=True)
            past, logits = out["past_key_values"], out["logits"][:, -1, :]

        return input_ids


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
def build_model():
    # for  v7
    return DecoderOnlyLM_RMSNorm_GQA(
    vocab_size=VOCAB_SIZE,         
    context_length=1536,
    n_embd=768, 
    n_layer=10,
    n_head=12, 
    n_kv_head=3,
    pad_id=PAD_ID,                 
    gradient_checkpointing=True,
) 
# for v2
#       return DecoderOnlyLM_RoPE(
#     vocab_size=VOCAB_SIZE,          
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v5
#       return DecoderOnlyLM_MoE_Sandwich(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     num_experts=2,
#     aux_loss_weight=0.01,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )
# for v6
#     return DecoderOnlyLM_HybridMask(
#     vocab_size=VOCAB_SIZE,         
#     context_length=1536,
#     n_embd=640,
#     n_layer=10,
#     n_head=10,
#     n_kv_head=2,
#     window_size=192,
#     global_every_n=4,
#     pad_id=PAD_ID,                  
#     gradient_checkpointing=True,
# )


HEAD_PATH   = "lm_head"      
LAYERS_PATH = "blocks"           
FINAL_NORM  = ("ln_f",)      
MODEL_PT    = "/kaggle/working/decoder_baseline_v7_final_model.pt"

In [ ]:
def parse_label_vocab(raw):
    if isinstance(raw, dict):
        if isinstance(raw.get("labels"), list):
            return [str(x) for x in raw["labels"]]
        if isinstance(raw.get("label2id"), dict):
            raw = raw["label2id"]
        if raw and all(isinstance(v, int) for v in raw.values()):
            return [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
        if raw and all(str(k).isdigit() for k in raw):
            return [raw[k] for k in sorted(raw, key=lambda x: int(x))]
        raise ValueError(f"Unrecognised label_vocab format: {list(raw)[:5]}")
    return [x if isinstance(x, str) else str(x) for x in raw]
 

def normalize_docs(raw, domain):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list):
            raw = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                if isinstance(v, dict):
                    items.append({"id": k, **v})
                elif isinstance(v, list):
                    items.append({"id": k, "text": v})
                else:
                    raise ValueError(f"unrecognised document value for id {k!r}: {type(v).__name__}")
            raw = items
    out = []
    for d in raw:
        text = d.get("text", d.get("sentences"))
        if text is None:
            raise ValueError(f"document {d.get('id')!r} has no 'text'/'sentences' field; keys: {list(d)}")
        sents = [x if isinstance(x, str) else str(x.get("text", x.get("sentence", ""))) for x in text]
        out.append({"key": f"{domain}/{d['id']}", "id": str(d["id"]), "domain": domain, "sents": sents,
                    "labels": list(d["labels"]) if d.get("labels") is not None else None})
    return out
 
 
def load_split(path, domain):
    return normalize_docs(json.load(open(path, encoding="utf-8")), domain)
 
 
def tokenize_docs(docs, encode_fn, clean_fn=None, max_sent_tokens=200, empty_id=0):
    out = []
    for d in docs:
        sent_ids = []
        for t in d["sents"]:
            t = clean_fn(t) if clean_fn else t
            r = encode_fn(t)
            r = r[0] if isinstance(r, tuple) else r
            ids = list(r)[:max_sent_tokens]
            sent_ids.append(ids if ids else [empty_id])
        out.append({**d, "sent_ids": sent_ids})
    return out
 
 
def rr_paths(root, domains=("CL", "IT"), include_test=False):
    for dm in domains:
        for split in ("train", "dev"):
            p[f"{dm}_{split}"] = f"{root}/Train/{dm}_{split}.json"
        if include_test:
            p[f"{dm}_test"] = f"{root}/Test/{dm}_test.json"
    return p
 
 
def download_rr_files(repo="Exploration-Lab/CS-781-Capstone", p3="Phase_3_RR", token=None, domains=("CL", "IT"),include_test=False):
    return {k: hf_hub_download(repo_id=repo, filename=f, repo_type="dataset", token=token) for k, f in rr_paths(p3, domains, include_test).items()}
 
 
def load_raw_docs(paths, split="test", domains=("CL", "IT")):
    return {dm: json.load(open(paths[f"{dm}_{split}"], encoding="utf-8")) for dm in domains}
 
def setup_data(paths, tokenizer_pkl, sep_token="</s>", clean=True, clean_fn=None,max_sent_tokens=200, domains=("CL", "IT")):
    if isinstance(paths, str):
        paths = rr_paths(paths, domains)
    bpe_data, encode = load_tokenizer(tokenizer_pkl)
    stoi = bpe_data["stoi"]
    pad_id = stoi["<pad>"]
    print("special tokens:", [t for t in stoi if isinstance(t, str)])
    sep_id = stoi.get(sep_token) if sep_token else None
    if sep_token and sep_id is None:
        print(f"WARNING: {sep_token!r} not in the vocabulary -> no marker token; each sentence is pooled at its last token")
    clean_fn = (clean_fn or clean_text) if clean else None
    empty_id = pad_id if sep_id is None else sep_id
 
    labels = parse_label_vocab(json.load(open(paths["labels"], encoding="utf-8")))
    label2id = {l: i for i, l in enumerate(labels)}
    D = {"labels": labels, "label2id": label2id, "pad_id": pad_id, "sep_id": sep_id, "empty_id": empty_id,"vocab_size": bpe_data["vocab_size"], "encode": encode, "clean": clean_fn,"max_sent_tokens": max_sent_tokens, "bpe_data": bpe_data}
    for split in ("train", "dev"):
        docs = []
        for dm in domains:
            docs += load_split(paths[f"{dm}_{split}"], dm)
        docs = tokenize_docs(docs, encode, clean_fn, max_sent_tokens, empty_id)
        for d in docs:
            bad = [l for l in d["labels"] if l not in label2id]
            assert not bad, f"labels not in label_vocab: {set(bad)} (doc {d['key']})"
            assert len(d["labels"]) == len(d["sents"]), f"labels/sentences length mismatch: {d['key']}"
            d["label_ids"] = [label2id[l] for l in d["labels"]]
        D[split] = docs
        by = Counter(d["domain"] for d in docs)
        n_s = sum(len(d["sents"]) for d in docs)
        tok = [sum(len(s) + 1 for s in d["sent_ids"]) for d in docs]
        print(f"{split:5s}: {len(docs)} docs {dict(by)}, {n_s} sentences, tokens/doc median {int(np.median(tok))} max {max(tok)}")
    cnt = Counter(l for d in D["train"] for l in d["labels"])
    print("labels:", labels)
    print("train label counts:", dict(cnt.most_common()))
    print(f"PAD_ID={pad_id}  VOCAB_SIZE={D['vocab_size']}  marker token id={sep_id}  cleaning={'on' if clean_fn else 'off'}")
    return D
 
 
def predict_documents(model, raw_docs, domain, D, CH, device, batch_size=8):
    docs = tokenize_docs(normalize_docs(raw_docs, domain), D["encode"], D["clean"], D["max_sent_tokens"], D["empty_id"])
    logp = predict_rr(model, docs, CH, D["sep_id"], D["pad_id"], device, batch_size=batch_size)
    return make_predictions(docs, logp, D["labels"]).get(domain, {})

In [ ]:
CH_SENT = dict(max_len=256, ctx_sents=0, ctx_tokens=0, owned_max=1)
CH_LEFT = dict(max_len=1024, ctx_sents=8, ctx_tokens=384, owned_max=10 ** 9)
CH_DOC = dict(max_len=1024, ctx_sents=10 ** 9, ctx_tokens=512, owned_max=10 ** 9)
 
def build_chunks(doc, max_len, ctx_sents, ctx_tokens, owned_max, sep_id):
    ids_list = doc["sent_ids"]
    n = len(ids_list)
    extra = 0 if sep_id is None else 1
    cost = [len(s) + extra for s in ids_list]
    assert not cost or max(cost) <= max_len, "a sentence is longer than max_len: lower max_sent_tokens"
    chunks, o = [], 0
    while o < n:
        c, used = o, 0
        while c > 0 and (o - c) < ctx_sents and used + cost[c - 1] <= ctx_tokens:
            c -= 1
            used += cost[c]
        while c < o and used + cost[o] > max_len:          
            used -= cost[c]
            c += 1
        e, total = o, used
        while e < n and (e - o) < owned_max and (e == o or total + cost[e] <= max_len):
            total += cost[e]
            e += 1
        toks, marks = [], []
        for k in range(c, e):
            toks += ids_list[k]
            if sep_id is not None:
                toks.append(sep_id)
            marks.append(len(toks) - 1)
        chunks.append({"input_ids": toks, "marks": marks, "c": c, "o": o, "e": e})
        o = e
    return chunks
 
 
def make_items(docs_tok, CH, sep_id, with_labels=True):
    items = []
    for di, d in enumerate(docs_tok):
        for ch in build_chunks(d, CH["max_len"], CH["ctx_sents"], CH["ctx_tokens"], CH.get("owned_max") or 10 ** 9, sep_id):
            if with_labels:
                lab = [-100] * (ch["o"] - ch["c"]) + d["label_ids"][ch["o"]:ch["e"]]
            else:
                lab = [0] * (ch["e"] - ch["c"])
            items.append({**ch, "labels": lab, "doc": di})
    return items
 
 
def collate_rr(batch, pad_id):
    B = len(batch)
    L = max(len(b["input_ids"]) for b in batch)
    S = max(len(b["marks"]) for b in batch)
    ids = torch.full((B, L), pad_id, dtype=torch.long)
    marks = torch.zeros((B, S), dtype=torch.long)
    mmask = torch.zeros((B, S), dtype=torch.bool)
    labels = torch.full((B, S), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n, s = len(b["input_ids"]), len(b["marks"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        marks[i, :s] = torch.tensor(b["marks"])
        mmask[i, :s] = True
        labels[i, :s] = torch.tensor(b["labels"])
    return {"input_ids": ids, "marks": marks, "mark_mask": mmask, "labels": labels}

In [ ]:
def load_phase1(build_fn, path):
    model = build_fn()
    ckpt = torch.load(path, map_location="cpu")
    sd = ckpt["model_state"] if isinstance(ckpt, dict) and "model_state" in ckpt else ckpt
    model.load_state_dict({k.replace("module.", "", 1): v for k, v in sd.items()}, strict=True)
    return model
 
 
class RRModel(nn.Module):
    def __init__(self, backbone, num_labels, head_path="lm_head", head="linear", dropout=0.1, lstm_hidden=256):
        super().__init__()
        self.backbone = backbone
        parent = backbone
        *parents, name = head_path.split(".")
        for p in parents:
            parent = getattr(parent, p)
        assert isinstance(getattr(parent, name), nn.Module), f"{head_path} not found"
        setattr(parent, name, nn.Identity())
        emb = next(m for m in backbone.modules() if isinstance(m, nn.Embedding))
        hidden = emb.embedding_dim
        self.vocab_size, self.num_labels = emb.num_embeddings, num_labels
        self.drop = nn.Dropout(dropout)
        self.head_type = head
        if head == "bilstm":
            self.seq = nn.LSTM(hidden, lstm_hidden, batch_first=True, bidirectional=True)
            self.classifier = nn.Linear(2 * lstm_hidden, num_labels)
        else:
            self.seq = None
            self.classifier = nn.Linear(hidden, num_labels)
 
    def head_parameters(self):
        ps = list(self.classifier.parameters())
        if self.seq is not None:
            ps += list(self.seq.parameters())
        return ps
 
    def forward(self, input_ids, marks, mark_mask):
        out = self.backbone(input_ids)
        h = out["logits"] if isinstance(out, dict) else out                     
        s = torch.gather(h, 1, marks.unsqueeze(-1).expand(-1, -1, h.size(-1)))  
        s = self.drop(s)
        if self.seq is not None:
            lengths = mark_mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(s, lengths, batch_first=True, enforce_sorted=False)
            o, _ = self.seq(packed)
            s, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=marks.size(1))
        return self.classifier(s)                                               
 
 
def dummy_batch(vocab_size, device, n_sent=4, seg=16):
    ids = torch.randint(0, vocab_size, (1, n_sent * seg), device=device)
    marks = torch.arange(seg - 1, n_sent * seg, seg, device=device).unsqueeze(0)
    return ids, marks, torch.ones_like(marks, dtype=torch.bool)
 
 
def trainable_state(model):
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
 
 
def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot
 

In [ ]:
def macro_scores(gold, pred, n_labels):
    gold, pred = np.asarray(gold), np.asarray(pred)
    per = {}
    for k in range(n_labels):
        tp = int(np.sum((gold == k) & (pred == k)))
        fp = int(np.sum((gold != k) & (pred == k)))
        fn = int(np.sum((gold == k) & (pred != k)))
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        F1 = 2 * P * R / (P + R) if P + R else 0.0
        per[k] = {"precision": P, "recall": R, "f1": F1, "support": tp + fn}
    scored = [k for k in per if per[k]["support"] > 0]           
    macro = {m: float(np.mean([per[k][m] for k in scored])) if scored else 0.0 for m in ("precision", "recall", "f1")}
    return {"per_label": per, "macro": macro}
 
 
@torch.no_grad()
def predict_rr(model, docs_tok, CH, sep_id, pad_id, device, batch_size=8, use_amp=True):
    model.eval()
    amp = use_amp and device.type == "cuda"
    items = make_items(docs_tok, CH, sep_id, with_labels=False)
    out = {d["key"]: np.zeros((len(d["sent_ids"]), model.num_labels), dtype=np.float32) for d in docs_tok}
    order = sorted(range(len(items)), key=lambda i: len(items[i]["input_ids"]))
    for s in range(0, len(order), batch_size):
        chunk = [items[i] for i in order[s:s + batch_size]]
        b = collate_rr(chunk, pad_id)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(b["input_ids"].to(device), b["marks"].to(device), b["mark_mask"].to(device))
        logp = F.log_softmax(logits.float(), dim=-1).cpu().numpy()
        for j, it in enumerate(chunk):
            k = docs_tok[it["doc"]]["key"]
            out[k][it["o"]:it["e"]] = logp[j, it["o"] - it["c"]: it["e"] - it["c"]]
    return out
 
 
def score_logp(docs_tok, logp, n_labels):
    gold, pred, dom = [], [], []
    for d in docs_tok:
        p = logp[d["key"]].argmax(-1).tolist()
        gold += d["label_ids"]; pred += p; dom += [d["domain"]] * len(p)
    overall = macro_scores(gold, pred, n_labels)
    by_dom = {}
    for dm in sorted(set(dom)):
        idx = [i for i, x in enumerate(dom) if x == dm]
        by_dom[dm] = macro_scores([gold[i] for i in idx], [pred[i] for i in idx], n_labels)
    return overall, by_dom
 
 
def evaluate_rr(model, docs_tok, CH, sep_id, pad_id, device, **kw):
    logp = predict_rr(model, docs_tok, CH, sep_id, pad_id, device, **kw)
    overall, by_dom = score_logp(docs_tok, logp, model.num_labels)
    return overall, by_dom, logp
 
 
def print_report(overall, by_dom, labels):
    print("overall macro:", {k: round(v, 4) for k, v in overall["macro"].items()})
    for dm, r in by_dom.items():
        print(f"  {dm}: macro F1 {r['macro']['f1']:.4f}  (P {r['macro']['precision']:.4f}, R {r['macro']['recall']:.4f})")
    for k, l in enumerate(labels):
        r = overall["per_label"][k]
        print(f"  {l:24s} F1 {r['f1']:.3f}  P {r['precision']:.3f}  R {r['recall']:.3f}  n={r['support']}")

In [ ]:
CFG_RR = dict(epochs=15, batch_size=2, grad_accum=8, lr_backbone=5e-4, lr_head=1e-3, weight_decay=0.01,warmup_ratio=0.1, max_grad_norm=1.0, fp16=True, class_weight=None)  
 
 
def class_weights(docs_tok, n_labels, mode):
    if not mode:
        return None
    cnt = np.bincount([i for d in docs_tok for i in d["label_ids"]], minlength=n_labels).astype(float) + 1.0
    w = cnt.sum() / (n_labels * cnt)
    if mode == "sqrt":
        w = np.sqrt(w)
    return torch.tensor(w / w.mean(), dtype=torch.float)
 
 
def train_rr(model, train_items, pad_id, cfg, device, val_fn=None, save_path=None, class_w=None):
    model.to(device)
    head_ids = {id(p) for p in model.head_parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids and p.requires_grad]
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if other:
        groups.append({"params": other, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
 
    loader = DataLoader(train_items, batch_size=cfg["batch_size"], shuffle=True,
                        collate_fn=lambda b: collate_rr(b, pad_id))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)
 
    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
 
    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    cw = class_w.to(device) if class_w is not None else None
    best, history = -1.0, []
    for epoch in range(cfg["epochs"]):
        model.train()
        run, nb, t0 = 0.0, 0, time.time()
        opt.zero_grad()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for i, b in enumerate(pbar):
            x, mk, mm, y = (b[k].to(device) for k in ("input_ids", "marks", "mark_mask", "labels"))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x, mk, mm)
            loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y.reshape(-1),
                                   weight=cw, ignore_index=-100)
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{run / nb:.4f}")
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        rec = {"epoch": epoch + 1, "train_loss": run / max(1, nb), "sec": round(time.time() - t0)}
        if val_fn is not None:
            rec["val"] = val_fn(model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val macro-F1={best:.4f})")
    return history

In [ ]:
def make_predictions(docs_tok, logp, labels):
    out = {}
    for d in docs_tok:
        preds = [labels[i] for i in logp[d["key"]].argmax(-1)]
        assert len(preds) == len(d["sents"]), f"length mismatch in {d['key']}"
        out.setdefault(d["domain"], {})[d["id"]] = preds
    return out
 
 
def write_rr_submission(preds_by_domain, out_dir="/kaggle/working", zip_name="submission_rr.zip"):
    files = []
    for dm, p in preds_by_domain.items():
        path = os.path.join(out_dir, f"{dm}_prediction.json")
        json.dump(p, open(path, "w", encoding="utf-8"))
        files.append(path)
    ids = [k for p in preds_by_domain.values() for k in p]
    if len(ids) == len(set(ids)):
        comb = os.path.join(out_dir, "prediction.json")
        json.dump({k: v for p in preds_by_domain.values() for k, v in p.items()}, open(comb, "w", encoding="utf-8"))
        files = [comb]
    else:
        print("WARNING: document ids collide across domains; writing per-domain files only")
    zp = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            z.write(f, arcname=os.path.basename(f))
    print(f"wrote {zp}: {[os.path.basename(f) for f in files]}, "
          f"{sum(len(p) for p in preds_by_domain.values())} docs")
 

In [ ]:
from peft import LoraConfig, get_peft_model

def show_linear_names(model):
    names = sorted({n.split(".")[-1] for n, m in model.backbone.named_modules() if isinstance(m, nn.Linear)})
    print("Linear module name suffixes:", names)
    return names

def apply_lora(model, r=32, alpha=64, dropout=0.05, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),use_rslora=False, use_dora=False):
    tm = targets if isinstance(targets, str) else list(targets)
    save = ["classifier"] + (["seq"] if getattr(model, "seq", None) is not None else [])
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=tm,
                     modules_to_save=save, bias="none", use_rslora=use_rslora, use_dora=use_dora)
    pm = get_peft_model(model, cfg)
    pm.print_trainable_parameters()
    for m in pm.modules():                                  
        if isinstance(getattr(m, "scaling", None), dict) and "default" in m.scaling:
            print(f"effective LoRA scaling: {m.scaling['default']:.4f}"f"({'alpha/sqrt(r), rsLoRA' if use_rslora else 'alpha/r'})")
            break
    return pm

def merge_lora(peft_model):
    return peft_model.merge_and_unload()

@torch.no_grad()
def check_lora_identity(build_fn, model_pt, head_path, n_labels, device, **kw):
    model = RRModel(load_phase1(build_fn, model_pt), n_labels, head_path=head_path).to(device).eval()
    batch = dummy_batch(model.vocab_size, device)
    ref = model(*batch)
    pm = apply_lora(model, **kw)
    pm.to(device).eval()
    print("max |diff| vs un-adapted model at init:", (pm(*batch) - ref).abs().max().item(), "(expect ~1e-5)")


def run_lora_rr(build_fn, model_pt, head_path, D, CH, device, tag="rr_lora", head="linear",r=32, alpha=64, dropout=0.1, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),cfg=None, eval_batch_size=8, use_rslora=False):
    cfg = cfg or CFG_RR
    model = RRModel(load_phase1(build_fn, model_pt), len(D["labels"]), head_path=head_path, head=head)
    model = apply_lora(model, r, alpha, dropout, targets, use_rslora=use_rslora)
    train_items = make_items(D["train"], CH, D["sep_id"], with_labels=True)
    print(f"train chunks: {len(train_items)}  (context setting: {CH})")

    def val_fn(m):
        overall, by_dom, _ = evaluate_rr(m, D["dev"], CH, D["sep_id"], D["pad_id"], device, batch_size=eval_batch_size)
        print("   dev:", {dm: round(r_["macro"]["f1"], 4) for dm, r_ in by_dom.items()})
        return overall["macro"]["f1"]

    cw = class_weights(D["train"], len(D["labels"]), cfg.get("class_weight"))
    hist = train_rr(model, train_items, D["pad_id"], cfg, device, val_fn=val_fn,save_path=f"/kaggle/working/rr_{tag}_best.pt", class_w=cw)
    return model, hist


def run_rslora_rr(build_fn, model_pt, head_path, D, CH, device, tag="rr_rslora", r=128, alpha=16, **kw):
    return run_lora_rr(build_fn, model_pt, head_path, D, CH, device, tag=tag, r=r, alpha=alpha, use_rslora=True, **kw)

In [ ]:
def model_size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

@torch.no_grad()
def quantize_linear_4bit(module, compute_dtype=torch.float16, quant_type="nf4", double_quant=True):
    try:
        import bitsandbytes as bnb
    except ImportError as e:
        raise ImportError("bitsandbytes is required for QLoRA") from e
    n = 0
    for parent in list(module.modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and not isinstance(child, bnb.nn.Linear4bit):
                new = bnb.nn.Linear4bit(child.in_features, child.out_features, bias=child.bias is not None,compute_dtype=compute_dtype, compress_statistics=double_quant,quant_type=quant_type)
                new.weight.data.copy_(child.weight.data.cpu())         
                new.weight.requires_grad = False
                if child.bias is not None:
                    new.bias.data.copy_(child.bias.data.cpu())
                    new.bias.requires_grad = False
                setattr(parent, cname, new)
                n += 1
    return n

@torch.no_grad()
def check_qlora_quant_error(build_fn, model_pt, head_path, n_labels, device, **kw):
    model = RRModel(load_phase1(build_fn, model_pt), n_labels, head_path=head_path).to(device).eval()
    batch = dummy_batch(model.vocab_size, device)
    ref = model(*batch)
    before = model_size_mb(model)
    quantize_linear_4bit(model.backbone, **kw)
    model.to(device).eval()
    out = model(*batch)
    print(f"fp32 {before:.0f} MB -> 4-bit {model_size_mb(model):.0f} MB (parameters)")
    print(f"logits: max |diff| {(out - ref).abs().max().item():.4f}, mean |diff| {(out - ref).abs().mean().item():.4f}, "
          f"mean |logit| {ref.abs().mean().item():.4f}   (diff should be small relative to the logits)")


def run_qlora_rr(build_fn, model_pt, head_path, D, CH, device, tag="rr_qlora", head="linear",r=32, alpha=64, dropout=0.05, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"),quant_type="nf4", double_quant=True, compute_dtype=torch.float16,cfg=None, eval_batch_size=8):
    cfg = cfg or CFG_RR
    model = RRModel(load_phase1(build_fn, model_pt), len(D["labels"]), head_path=head_path, head=head)
    print(f"fp32 model: {model_size_mb(model):.0f} MB of parameters")
    n = quantize_linear_4bit(model.backbone, compute_dtype, quant_type, double_quant)
    model.to(device)                                             
    print(f"quantized {n} Linear layers to {quant_type} (double_quant={double_quant})"f"model now {model_size_mb(model):.0f} MB of parameters")
    model = apply_lora(model, r, alpha, dropout, targets)         
    train_items = make_items(D["train"], CH, D["sep_id"], with_labels=True)
    print(f"train chunks: {len(train_items)}  (context setting: {CH})")

    def val_fn(m):
        overall, by_dom, _ = evaluate_rr(m, D["dev"], CH, D["sep_id"], D["pad_id"], device, batch_size=eval_batch_size)
        print("   dev:", {dm: round(r_["macro"]["f1"], 4) for dm, r_ in by_dom.items()})
        return overall["macro"]["f1"]

    cw = class_weights(D["train"], len(D["labels"]), cfg.get("class_weight"))
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    hist = train_rr(model, train_items, D["pad_id"], cfg, device, val_fn=val_fn,
                    save_path=f"/kaggle/working/rr_{tag}_best.pt", class_w=cw)
    if device.type == "cuda":
        print(f"peak GPU memory during training: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    return model, hist

In [ ]:
_SHIFT_IDX = {}

def _shift_index(d, device):
    key = (d, str(device))
    if key not in _SHIFT_IDX:
        i = torch.arange(d, device=device)
        _SHIFT_IDX[key] = (i[None, :] - i[:, None]) % d
    return _SHIFT_IDX[key]


class LoRMALinear(nn.Module):
    def __init__(self, base, variant="plus", r=8, alpha=16, dropout=0.0):
        super().__init__()
        assert variant in ("plus", "pi") and isinstance(base, nn.Linear)
        self.base, self.variant, self.r = base, variant, r
        d = base.out_features
        self.scale = alpha / r if variant == "plus" else 1.0     
        self.A = nn.Parameter(torch.zeros(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        for p in self.base.parameters():
            p.requires_grad = False
        self.reset_parameters()

    @torch.no_grad()
    def reset_parameters(self):
        if self.variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
            self.B.zero_()
        else:
            self.A.zero_(); self.A[0, 0] = 1.0
            self.B.zero_(); self.B[:, 0] = 1.0
            if self.r > 1:
                nn.init.kaiming_uniform_(self.B[:, 1:], a=math.sqrt(5))

    def _pi_matrix(self):
        BA = self.B @ self.A
        return torch.gather(BA, 1, _shift_index(BA.size(0), BA.device))

    def forward(self, x):
        y = F.linear(x, self.base.weight)                          
        if self.variant == "plus":
            y = y + self.scale * F.linear(F.linear(self.drop(y), self.A), self.B)   
        else:
            y = y @ self._pi_matrix().t().to(y.dtype)              
        if self.base.bias is not None:
            y = y + self.base.bias
        return y

    @torch.no_grad()
    def merged_weight(self):                                       
        if self.variant == "plus":
            d = self.A.size(1)
            M = torch.eye(d, device=self.A.device, dtype=self.A.dtype) + self.scale * (self.B @ self.A)
        else:
            M = self._pi_matrix()
        return M @ self.base.weight

    @torch.no_grad()
    def to_linear(self):
        W = self.merged_weight()
        lin = nn.Linear(self.base.in_features, self.base.out_features,
                        bias=self.base.bias is not None).to(W.device, W.dtype)
        lin.weight.copy_(W)
        if self.base.bias is not None:
            lin.bias.copy_(self.base.bias)
        return lin

    def extra_repr(self):
        return f"variant={self.variant}, r={self.r}, d={self.A.size(1)}"


def inject_lorma(root, targets=("q_proj", "v_proj"), variant="plus", r=8, alpha=16, dropout=0.0):
    names = [n for n, m in root.named_modules()
             if isinstance(m, nn.Linear) and any(n == t or n.endswith("." + t) for t in targets)]
    assert names, f"no nn.Linear matched {targets}"
    for n in names:
        parent_path, _, child = n.rpartition(".")
        parent = root.get_submodule(parent_path) if parent_path else root
        setattr(parent, child, LoRMALinear(getattr(parent, child), variant, r, alpha, dropout))
    print(f"LoRMA-{variant}: wrapped {len(names)} layers, e.g. {names[:3]}")
    return names


def freeze_for_lorma(model):
    for p in model.parameters():
        p.requires_grad = False
    for m in model.modules():
        if isinstance(m, LoRMALinear):
            m.A.requires_grad = True
            m.B.requires_grad = True
    for p in model.head_parameters():
        p.requires_grad = True


@torch.no_grad()
def merge_lorma(root):
    for _, m in list(root.named_modules()):
        for cn, c in list(m.named_children()):
            if isinstance(c, LoRMALinear):
                setattr(m, cn, c.to_linear())


def run_lorma_rr(build_fn, ckpt, head_path, D, CH, device, tag="rr_lorma", variant="plus", r=16, alpha=32,targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"), dropout=0.1, head="linear", cfg=None):
    cfg = cfg or CFG_RR
    backbone = load_phase1(build_fn, ckpt)
    model = RRModel(backbone, len(D["labels"]), head_path=head_path, head=head)
    inject_lorma(model.backbone, targets, variant, r, alpha, dropout)
    freeze_for_lorma(model)
    count_params(model)
    train_items = make_items(D["train"], CH, D["sep_id"])
    val_fn = lambda m: evaluate_rr(m, D["dev"], CH, D["sep_id"], D["pad_id"], device)[0]["macro"]["f1"]
    cw = class_weights(D["train"], len(D["labels"]), cfg.get("class_weight"))
    hist = train_rr(model, train_items, D["pad_id"], cfg, device, val_fn=val_fn,
                    save_path=f"/kaggle/working/{tag}_best.pt", class_w=cw)
    return model, hist



def check_lorma(build_fn, ckpt, targets=("q_proj", "k_proj", "v_proj", "c_proj", "c_fc"), variant="plus", r=16, alpha=32):
    ids = torch.randint(0, 1000, (1, 64))
    plain = load_phase1(build_fn, ckpt).eval()
    wrapped = load_phase1(build_fn, ckpt)
    inject_lorma(wrapped, targets, variant, r, alpha)
    wrapped.eval()
    with torch.no_grad():
        d0 = (plain(ids)["logits"] - wrapped(ids)["logits"]).abs().max().item()
        for m in wrapped.modules():
            if isinstance(m, LoRMALinear):
                nn.init.normal_(m.A, std=0.02); nn.init.normal_(m.B, std=0.02)
        before = wrapped(ids)["logits"]
        merge_lorma(wrapped)
        d1 = (before - wrapped(ids)["logits"]).abs().max().item()
    print(f"identity at init: max|diff| = {d0:.2e} (expect ~0) | merged vs factored: max|diff| = {d1:.2e} (expect ~1e-5)")


def check_lorma_grads(model, device):
    model.to(device).train()
    ids, marks, mm = dummy_batch(model.vocab_size, device)
    model(ids, marks, mm).sum().backward()
    bad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
    print("trainable params without a gradient:", bad[:5] if bad else "none")

In [ ]:
def student_by_truncation(build_fn, model_pt, head_path, n_labels, keep_layers=5, head="linear", blocks_attr="blocks"):
    bb = load_phase1(build_fn, model_pt)
    blocks = getattr(bb, blocks_attr)
    n = len(blocks)
    idx = sorted(set(int(round(i)) for i in np.linspace(0, n - 1, keep_layers)))
    setattr(bb, blocks_attr, nn.ModuleList([blocks[i] for i in idx]))
    print(f"student keeps blocks {idx} of {n}")
    return RRModel(bb, n_labels, head_path=head_path, head=head)


def student_from_model(student_build_fn, student_pt, head_path, n_labels, head="linear"):
    return RRModel(load_phase1(student_build_fn, student_pt), n_labels, head_path=head_path, head=head)

@torch.no_grad()
def teacher_doc_logp(teachers, D, device, split="train", batch_size=8):
    acc = None
    for t in teachers:
        model, CH, *rest = t
        Dt = rest[0] if rest else D
        lp = predict_rr(model, Dt[split], CH, Dt["sep_id"], Dt["pad_id"], device, batch_size=batch_size)
        acc = lp if acc is None else {k: acc[k] + lp[k] for k in acc}
    return {k: v / len(teachers) for k, v in acc.items()}


def make_kd_items(docs_tok, CH, sep_id, teacher_lp):
    items = make_items(docs_tok, CH, sep_id, with_labels=True)
    for it in items:
        it["soft"] = teacher_lp[docs_tok[it["doc"]]["key"]][it["c"]:it["e"]]
    return items


def collate_kd_rr(batch, pad_id):
    b = collate_rr(batch, pad_id)
    B, S = b["labels"].shape
    soft = torch.zeros((B, S, batch[0]["soft"].shape[1]))
    for i, it in enumerate(batch):
        soft[i, :len(it["soft"])] = torch.from_numpy(np.asarray(it["soft"], dtype=np.float32))
    b["soft"] = soft
    return b

def kd_loss(logits, y, soft_logp, T, kd_alpha, class_w=None):
    C = logits.size(-1)
    ce = F.cross_entropy(logits.reshape(-1, C), y.reshape(-1), weight=class_w, ignore_index=-100)
    m = (y != -100)                                                
    s_logp = F.log_softmax(logits / T, dim=-1)
    t_prob = F.softmax(soft_logp / T, dim=-1)                        
    kl = (t_prob * (torch.log(t_prob.clamp_min(1e-9)) - s_logp)).sum(-1)
    kd = (kl * m).sum() / m.sum().clamp(min=1) * (T * T)
    return kd_alpha * ce + (1 - kd_alpha) * kd, ce.detach(), kd.detach()


def train_rr_kd(model, items, pad_id, cfg, device, val_fn=None, save_path=None, class_w=None, T=2.0, kd_alpha=0.5):
    model.to(device)
    head_ids = {id(p) for p in model.head_parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids and p.requires_grad]
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": cfg["lr_head"]}]
    if other:
        groups.append({"params": other, "lr": cfg["lr_backbone"]})
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

    loader = DataLoader(items, batch_size=cfg["batch_size"], shuffle=True, collate_fn=lambda b: collate_kd_rr(b, pad_id))
    total_steps = max(1, (len(loader) // cfg["grad_accum"]) * cfg["epochs"])
    warmup = int(cfg["warmup_ratio"] * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = cfg["fp16"] and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    cw = class_w.to(device) if class_w is not None else None
    best, history = -1.0, []
    for epoch in range(cfg["epochs"]):
        model.train()
        run, run_ce, run_kd, nb, t0 = 0.0, 0.0, 0.0, 0, time.time()
        opt.zero_grad()
        pbar = tqdm(loader, desc=f"epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for i, b in enumerate(pbar):
            x, mk, mm, y = (b[k].to(device) for k in ("input_ids", "marks", "mark_mask", "labels"))
            soft = b["soft"].to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x, mk, mm)
            loss, ce, kd = kd_loss(logits.float(), y, soft, T, kd_alpha, cw)
            scaler.scale(loss / cfg["grad_accum"]).backward()
            run += loss.item(); run_ce += ce.item(); run_kd += kd.item(); nb += 1
            pbar.set_postfix(loss=f"{run / nb:.4f}")
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad()
        nb = max(1, nb)
        rec = {"epoch": epoch + 1, "train_loss": run / nb, "ce": run_ce / nb, "kd": run_kd / nb, "sec": round(time.time() - t0)}
        if val_fn is not None:
            rec["val"] = val_fn(model)
            if rec["val"] > best:
                best = rec["val"]
                if save_path:
                    torch.save(trainable_state(model), save_path)
        history.append(rec)
        print(rec)
    if val_fn is not None and save_path:
        model.load_state_dict(torch.load(save_path, map_location=device), strict=False)
        print(f"restored best epoch (val macro-F1={best:.4f})")
    return history

CFG_KD = {**CFG_RR, "lr_backbone": 1e-4, "lr_head": 1e-3, "weight_decay": 0.01, "epochs": 15}   

def run_kd_rr(teachers, student, D, CH, device, tag="rr_kd", T=2.0, kd_alpha=0.5, student_mode="full",D_student=None, lora_kw=None, cfg=None, eval_batch_size=8):
    cfg = cfg or CFG_KD
    Ds = D_student or D
    teacher_lp = teacher_doc_logp(teachers, D, device, "train")
    items = make_kd_items(Ds["train"], CH, Ds["sep_id"], teacher_lp)
    print(f"KD items: {len(items)} chunks, T={T}, kd_alpha={kd_alpha}, student_mode={student_mode}")
    if student_mode == "lora":
        student = apply_lora(student, **(lora_kw or {}))
    else:
        for p in student.parameters():
            p.requires_grad = True
        count_params(student)

    def val_fn(m):
        overall, by_dom, _ = evaluate_rr(m, Ds["dev"], CH, Ds["sep_id"], Ds["pad_id"], device, batch_size=eval_batch_size)
        print("   dev:", {dm: round(r_["macro"]["f1"], 4) for dm, r_ in by_dom.items()})
        return overall["macro"]["f1"]

    cw = class_weights(Ds["train"], len(Ds["labels"]), cfg.get("class_weight"))
    hist = train_rr_kd(student, items, Ds["pad_id"], cfg, device, val_fn=val_fn,
                       save_path=f"/kaggle/working/rr_{tag}_best.pt", class_w=cw, T=T, kd_alpha=kd_alpha)
    return student, hist


def kd_report(models, device, n_docs=None):
    for name, (m, CH, Dm) in models.items():
        docs = Dm["dev"][:n_docs] if n_docs else Dm["dev"]
        n_par = sum(p.numel() for p in m.parameters())
        mb = sum(p.numel() * p.element_size() for p in m.parameters()) / 1e6
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.time()
        logp = predict_rr(m, docs, CH, Dm["sep_id"], Dm["pad_id"], device)
        if device.type == "cuda":
            torch.cuda.synchronize()
        sec = time.time() - t0
        overall, by_dom = score_logp(docs, logp, m.num_labels)
        print(f"{name:10s} params {n_par/1e6:7.2f}M ({mb:6.0f} MB) | dev macro-F1 {overall['macro']['f1']:.4f} "
              f"{ {dm: round(r_['macro']['f1'], 4) for dm, r_ in by_dom.items()} } | dev inference {sec:.0f}s")

def load_lora_teacher(build_fn, model_pt, head_path, D, ckpt_path, device, head="linear", **lora_kw):
    model = RRModel(load_phase1(build_fn, model_pt), len(D["labels"]), head_path=head_path, head=head)
    model = apply_lora(model, **lora_kw)
    sd = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(sd, strict=False)
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    print(f"loaded {len(sd)} tensors; trainable params missing from the checkpoint: {len(trainable - set(sd))}")
    return model.to(device).eval()

In [ ]:
P3 = "Phase_3_RR"
hf_dataset_repo = "Exploration-Lab/CS-781-Capstone"
p3 = download_rr_files(hf_dataset_repo, P3, HF_TOKEN,include_test=True)
D = setup_data(p3, "/kaggle/working/legal_bpe_tokenizer.pkl",sep_token="</s>")
PAD_ID, VOCAB_SIZE = D["pad_id"], D["vocab_size"]        

In [ ]:
model_lora, hist_lora = run_lora_rr(build_model, MODEL_PT, HEAD_PATH, D, CH_LEFT, device, tag="rr_lora")

In [ ]:
!pip install -U bitsandbytes

In [ ]:
model_qlora, hist_qlora = run_qlora_rr(build_model, MODEL_PT, HEAD_PATH, D, CH_LEFT, device, tag="rr_qlora")

In [ ]:
model_lorma, hist_lorma = run_lorma_rr(build_model,MODEL_PT,HEAD_PATH,D,CH_LEFT,device,tag="rr_lorma")

In [ ]:

teacher = load_lora_teacher(build_model, MODEL_PT, HEAD_PATH, D,"/kaggle/working/rr_rr_lora_best.pt", device, r=32, alpha=64)
student = student_by_truncation(build_model, MODEL_PT, HEAD_PATH, len(D["labels"]), keep_layers=5)

student, hist_kd = run_kd_rr([(teacher, CH_LEFT)],student,D,CH_LEFT, device,tag="rr_kd")
kd_report({"teacher": (teacher, CH_LEFT, D), "student": (student, CH_LEFT, D)}, device)

In [ ]:
model_rs, hist_rs = run_rslora_rr(build_model, MODEL_PT, HEAD_PATH, D, CH_LEFT, device, tag="rr_rslora")

In [ ]:
raw = load_raw_docs(p3, "test")
preds = {dm: predict_documents(model_lora, docs, dm, D, CH_LEFT, device) for dm, docs in raw.items()}
assert all(len(preds[dm][d["id"]]) == len(d["sents"]) for dm, docs in raw.items() for d in normalize_docs(docs, dm))
write_rr_submission(preds, zip_name="submission_rr_mv6_lora.zip")